# Multi-Query RAG

<img src="../images/Multi-Query RAG.png" width=“500”>

In [1]:
from langchain_core.globals import set_debug
set_debug(False)

## Initialize Azure Chat OpenAI

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

load_dotenv(override=True, verbose=True)

chat_llm = AzureChatOpenAI(
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_deployment=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')
)

## Initialize Azure Embedding Open AI

In [3]:
emd_llm = AzureOpenAIEmbeddings(
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_deployment=os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
)

## Load the Document and upload it to vector store

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader

file = '../Data/HTTP-Status-Codes.pdf'
pdf_docs = PyMuPDFLoader(file_path=file).load()


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 50
)
chunks = splitter.split_documents(documents=pdf_docs)

In [6]:
from langchain_community.vectorstores import FAISS

faiss = FAISS.from_documents(documents=chunks, embedding=emd_llm)


## Multi Query generation from User's query

### Multi query Prompt

In [7]:
MULTI_QUERY_PROMPT = """"
You are a helpful assistant that generates alternative phrasings of a user's question to improve document retrieval from a vector database.
Produce {num_queries} different versions that capture distinct phrasings and perspectives of the question.
Important: Generate one query per line, output ONE query per line, with no numbering, bullets, or extra text.
"""

In [8]:
from langchain_core.prompts import ChatPromptTemplate

multi_query_prompt_template = ChatPromptTemplate.from_messages(
    [
    ("system", MULTI_QUERY_PROMPT),
    ("user", "Original questions: {question}"),
    ]
)

In [9]:
num_queries = 3
question = "What is error code 404?"
prompt = multi_query_prompt_template.invoke({"num_queries": num_queries, "question": question})

### Invoke Chat LLM to generate the queries

In [10]:
response = chat_llm.invoke(prompt)
print(response.content)

What does HTTP 404 Not Found mean?

Why does a website return error code 404 and how can I fix it?

Explain the common causes and troubleshooting steps for a 404 error on a web page


In [11]:
multiple_queries = [q for q in response.content.split("\n")]
print(f"total queries generated: {len(multiple_queries)}")
for q in multiple_queries:
    print(q)

total queries generated: 5
What does HTTP 404 Not Found mean?

Why does a website return error code 404 and how can I fix it?

Explain the common causes and troubleshooting steps for a 404 error on a web page


## Call to Vector store to get the relevant chunks

In [12]:
fetched_response = []
for q in multiple_queries:
    similar_docs = faiss.similarity_search(
        query=q, 
        k = 3)
    print(f"query: {q}", end="\n")
    for sd in similar_docs:
        print(sd.page_content, end="\n\n")
        fetched_response.append(sd.page_content)
    print("\n------\n")

query: What does HTTP 404 Not Found mean?
restricted unless you get different credentials with proper authorization.
404 Not Found
The server can’t find the requested resource. No forwarding address. In

200 OK when it works or 404 Not Found when the resource doesn’t exist.
How to Check HTTP Status Code Using Tools
You need to see status codes when debugging. Several tools make this easy.

404 Not Found Error Impact
410 Gone vs 404 Error Codes
500 Internal Server Error Solutions
503 Service Unavailable Fix
Common HTTP Status Code FAQ
Search...
Tutorials
Comparisons
News
Keep in touch


------

query: 
fetched the page. POST submitted your form. PUT updated the resource.
DELETE removed it. This is the code you want to see.
201 Created

Sends data to a resource. Submits forms. Creates resources. Not idempotent.
Can be cacheable under certain conditions.
PUT
Replaces a resource or its representation entirely. Idempotent. Running it

418 I’m a Teapot
An April Fools’ joke from the Hyper Tex

In [13]:
PROMPT_TEMPLATE = """"
You are a helpful assistance. You answer user's query {question} from the provided context {context}. If context isn't sufficient to provide the answer, inform the user that Context isn't sufficient to provide the answer of your query.
"""
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", PROMPT_TEMPLATE),
        ("user",  "user query: {question}"),
    ]
)


In [14]:
prompt =  prompt_template.invoke({'question': question, 'context': fetched_response})
response = chat_llm.invoke(prompt)
print(response)

print(response.content)

content='A 404 is an HTTP status code meaning "404 Not Found." It tells you the server can’t find the requested resource (the page or file) — a client-side (4xx) error. \n\nCommon causes:\n- Wrong or mistyped URL\n- The page was deleted or moved\n- Permission/authorization issues\n- Server or configuration problems (e.g., broken links, bad .htaccess or plugins on WordPress)\n\nImpact and fixes (brief):\n- Search engines may remove the page from their index, causing lost traffic and lost link equity.\n- For deleted pages, redirect to relevant content with a 301.\n- If the page should exist, check the URL, permissions, server configuration, plugins or .htaccess, and use developer tools or crawling software to find the problem.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 619, 'prompt_tokens': 717, 'total_tokens': 1336, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_predic